# Pinecone을 활용한 RAG
## pinecone과 chroma의 차이
- chroma는 inmemory인 반면 pinecone은 cloud를 활용한다.
- cloud를 사용하기 때문에 서버 이전 시에도 편리하다.

In [ ]:
%pip install docx2txt langchain-community --quiet

In [ ]:
%pip install langchain-text-splitters --quiet

In [ ]:
%pip install langchain-upstage python-dotenv

In [5]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
)

loader = Docx2txtLoader("./data/tax.docx")
document_list = loader.load_and_split(text_splitter=text_splitter)

In [17]:
from dotenv import load_dotenv

load_dotenv()

True

In [7]:
from langchain_upstage import UpstageEmbeddings

embedding = UpstageEmbeddings(model="solar-embedding-1-large")

## pinecone 활용한 벡터 DB 저장

In [ ]:
%pip install --upgrade --quiet langchain-pinecone \
     langchain-openai langchain 


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [18]:
import os
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

index_name = "tax-index"
pinecone_api_key = os.getenv("PINECONE_API_KEY")

pc = Pinecone(api_key=pinecone_api_key)

database = PineconeVectorStore.from_documents(document_list, embedding, index_name=index_name)

* 인덱스 생성 필요 - 현재 과제에서는 pinecone console에서 진행함

In [19]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o")

In [21]:
from langchain_core.prompts import ChatPromptTemplate

query = "연봉 5000만원 직장이 기준 소득세는 얼마인가요?"

prompt = ChatPromptTemplate.from_messages([
    ('system', """소득세법에 대한 질문에 아래 context를 근거로 답변하세요.
질문에 대한 직접적인 문장이 없더라도, 공제 규정과 세율표를 활용해 단계적으로 계산할 수 있다면 계산 과정을 보여주고 답하세요.
context에서 전혀 근거를 찾을 수 없을 때만 모른다고 답하세요.

    [Context]
    {context}
    """),
    ('human', '{question}')
])

In [24]:
from langchain_classic.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever=database.as_retriever(),
    chain_type_kwargs={"prompt": prompt}
)

ai_message = qa_chain.invoke({"query": query})

In [26]:
ai_message['result']

'연봉 5000만원의 직장인이 부담하는 소득세를 계산하기 위해서는 다음의 과정이 필요합니다.\n\n1. **종합소득과세표준**을 계산합니다. 연봉이 5000만원인 경우, 종합소득과세표준은 총급여액에서 근로소득공제를 차감하여 계산합니다. 근로소득공제는 총급여액에 적용되는 일정 비율과 금액에 따라 계산됩니다.\n\n2. **근로소득공제**:\n    - 5000만원의 총급여액에 대해 적용되는 근로소득공제를 계산해야 합니다. 그러나 context 내에서 근로소득공제에 대한 세부적 정보를 제공하지 않았습니다. 따라서 일반적 기준을 사용합니다\n    - 예시로, 총급여 4000만원을 초과하고 8000만원 이하인 경우, 1100만원 + (초과금액의 20%)을 공제합니다.\n    - 따라서, 공제액 = 1100만원 + (5000만원 - 4000만원) * 20% = 1100만원 + 200만원 = 1300만원\n\n3. **과세표준**:\n    - 과세표준 = 총급여 - 근로소득공제 = 5000만원 - 1300만원 = 3700만원\n\n4. **소득세 계산**:\n    - 과세표준에 따른 세율을 적용합니다. context 내 세율표는 없으나 일반적인 세율을 적용하면:\n    - 예시: 1200만원 이하: 6%, 1200만원 초과 4600만원 이하: 15%\n    - 1200만원까지의 세액 = 1200만원 * 6% = 72만원\n    - 1200만원 초과분(2500만원)에 대한 세액 = 2500만원 * 15% = 375만원\n    - 총 세액 = 72만원 + 375만원 = 447만원\n\n5. **세액공제 등을 통한 최종 소득세 산출**:\n    - 추가적인 세율공제가 없다면 산출된 총세액이 소득세가 됩니다.\n    - 연금계좌세액공제, 자녀세액공제 등의 다른 세액공제가 있을 경우 이를 차감하여 최종 소득세를 구합니다.\n\n이와 같은 계산을 통해, 연봉 5000만원인 경우의 소득세를 계산할 수 있습니다. 다만, 위 계산에서 생략된 다른 공제나 부분들이 적용될